In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_8.py ──
"""
Shared infrastructure for MLFP04 Exercise 8 — Deep Learning Foundations.

Contains: synthetic XOR data, synthetic Singapore-medical image data,
reusable training loops, gradient monitoring helpers, ModelVisualizer
output paths. Technique-specific code (model classes, per-file training
loops, scenario narratives) does NOT belong here — it lives per file.
"""

from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from kailash_ml import ModelVisualizer


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT — seeds, device, output dir
# ════════════════════════════════════════════════════════════════════════
setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex8_deep_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared hyperparameters
N_FEATS_XOR = 4
N_XOR_SAMPLES = 200
N_IMG_SAMPLES = 5000
IMG_SIZE = 64
N_CHANNELS = 1
N_CLASSES = 5
BATCH_SIZE = 64

# Kailash visualiser (used by every phase 4 block)
viz = ModelVisualizer()

# ════════════════════════════════════════════════════════════════════════
# DATA — XOR toy problem (Tasks 1-3)
# ════════════════════════════════════════════════════════════════════════


def make_xor_data(
    n_samples: int = N_XOR_SAMPLES, n_features: int = N_FEATS_XOR, seed: int = 42
) -> tuple[torch.Tensor, torch.Tensor, np.ndarray]:
    """Generate a synthetic XOR classification task.

    Label is XOR of the sign of features 0 and 1. Features 2..n-1 are noise.
    Returns (X_tensor, y_tensor, y_numpy) on CPU.
    """
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n_samples, n_features)).astype(np.float32)
    y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(np.float32)
    X_t = torch.from_numpy(X)
    y_t = torch.from_numpy(y).unsqueeze(1)
    return X_t, y_t, y


# ════════════════════════════════════════════════════════════════════════
# DATA — Synthetic Singapore Hospital imaging tensors (Tasks 4-10)
# ════════════════════════════════════════════════════════════════════════
# Scenario: NUH (National University Hospital) chest-film triage. The real
# pipeline uses anonymised 512x512 DICOMs; this exercise uses 64x64 random
# tensors with the same multi-label structure so training completes in
# minutes on a laptop CPU / Colab T4.

SG_HOSPITAL_CLASSES = [
    "pneumonia",
    "effusion",
    "atelectasis",
    "nodule",
    "normal",
]


def make_sg_imaging_data(
    n_samples: int = N_IMG_SAMPLES, seed: int = 42
) -> tuple[np.ndarray, np.ndarray]:
    """Return (X_images, y_labels) as float32 numpy arrays.

    X: (N, 1, 64, 64) — simulated single-channel chest film tensors.
    y: (N, 5) — multi-label (~15% positive per class).
    """
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n_samples, N_CHANNELS, IMG_SIZE, IMG_SIZE)).astype(
        np.float32
    )
    y = (rng.random((n_samples, N_CLASSES)) > 0.85).astype(np.float32)
    return X, y


def build_sg_loaders(
    batch_size: int = BATCH_SIZE,
) -> tuple[DataLoader, DataLoader, np.ndarray, np.ndarray]:
    """Produce (train_loader, test_loader, X_test_np, y_test_np) for the CNN tasks."""
    X, y = make_sg_imaging_data()
    split = int(0.8 * len(X))
    X_tr, X_te = X[:split], X[split:]
    y_tr, y_te = y[:split], y[split:]

    train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
    test_ds = TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)
    return train_loader, test_loader, X_te, y_te


# ════════════════════════════════════════════════════════════════════════
# TRAINING UTILITIES
# ════════════════════════════════════════════════════════════════════════


def train_xor_net(
    net: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    optimiser: torch.optim.Optimizer,
    n_epochs: int = 100,
    criterion: nn.Module | None = None,
) -> list[float]:
    """Fit a small binary classifier to XOR data. Returns per-epoch loss."""
    crit = criterion or nn.BCEWithLogitsLoss()
    losses: list[float] = []
    for _ in range(n_epochs):
        optimiser.zero_grad()
        loss = crit(net(X), y)
        loss.backward()
        optimiser.step()
        losses.append(loss.item())
    return losses


def xor_accuracy(net: nn.Module, X: torch.Tensor, y_np: np.ndarray) -> float:
    """Binary accuracy on XOR data (threshold at 0.5)."""
    net.eval()
    with torch.no_grad():
        probs = torch.sigmoid(net(X)).numpy().flatten()
    return float(((probs > 0.5) == y_np).mean())


def grad_norm(model: nn.Module) -> float:
    """L2 norm of the concatenated gradient vector."""
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.data.norm(2).item() ** 2
    return float(total**0.5)


def train_cnn_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimiser: torch.optim.Optimizer,
    criterion: nn.Module,
    clip_value: float | None = None,
) -> tuple[float, float]:
    """Train for one epoch on the Singapore imaging loader.

    Returns (mean_loss, mean_grad_norm). If ``clip_value`` is set, the grad
    norm is measured pre-clipping and ``clip_grad_norm_`` is applied.
    """
    model.train()
    losses: list[float] = []
    grads: list[float] = []
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimiser.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        grads.append(grad_norm(model))
        if clip_value is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_value)
        optimiser.step()
        losses.append(loss.item())
    return float(np.mean(losses)), float(np.mean(grads))


def eval_cnn(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> float:
    """Return mean validation loss across the loader."""
    model.eval()
    losses: list[float] = []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            losses.append(criterion(model(X_b), y_b).item())
    return float(np.mean(losses))


# ════════════════════════════════════════════════════════════════════════
# CNN BUILDING BLOCKS (reused across files 03, 04, 05)
# ════════════════════════════════════════════════════════════════════════


class ResBlock(nn.Module):
    """Residual block: skip connection preserves gradient flow."""

    def __init__(self, channels: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # noqa: D401
        residual = x
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(out + residual)


class TriageCNN(nn.Module):
    """CNN for multi-label Singapore hospital triage.

    Architecture: Conv32 -> ResBlock -> Conv64 -> ResBlock -> AdaptiveAvgPool
    -> Dropout -> Linear. Designed for the multi-label BCEWithLogitsLoss
    setup used throughout Exercise 8.
    """

    def __init__(self, n_classes: int = N_CLASSES, dropout_rate: float = 0.3) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            ResBlock(32),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            ResBlock(64),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # noqa: D401
        return self.classifier(self.features(x))


def count_params(model: nn.Module) -> tuple[int, int]:
    """Return (total_params, trainable_params)."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# ════════════════════════════════════════════════════════════════════════
# DATA LOADER ENTRY POINT
# ════════════════════════════════════════════════════════════════════════
# We expose an MLFPDataLoader handle so student files have a single import
# path even though the tensors are generated on the fly. Real datasets for
# CNN fine-tuning live in Module 5.
loader = MLFPDataLoader()




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 8.5: Regularisation, Clipping, Early Stop, ONNX Export
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Contrast baseline vs dropout vs batch-norm vs both
#   - Apply gradient clipping to prevent exploding gradients
#   - Implement early stopping on validation loss
#   - Export the trained CNN to ONNX via kailash-ml OnnxBridge
#
# PREREQUISITES: 04_optimisers_schedulers.py
#
# ESTIMATED TIME: ~45 min
#
# TASKS:
#   1. Theory — regularisation tricks compound
#   2. Build — four regularisation variants + EarlyStopping class
#   3. Train — full loop with clipping, scheduler, early stop
#   4. Visualise — val loss across variants + final gradient norms
#   5. Apply — DSO Labs Singapore: ONNX export cut inference cost by 7x
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.optim as optim

from kailash_ml.bridge.onnx_bridge import OnnxBridge


print("\n" + "=" * 70)
print("  Regularisation, Clipping, Early Stopping, ONNX Export")
print("=" * 70)



## THEORY — Compounding regularisers


In [ ]:
# Dropout breaks co-adaptation. BatchNorm stabilises the loss landscape.
# Weight decay shrinks weights. Gradient clipping caps updates. Early
# stopping uses training-time itself as a regulariser. Combined, they
# make training robust without heavy per-task tuning.

train_loader, test_loader, X_test_np, y_test_np = build_sg_loaders()
criterion = nn.BCEWithLogitsLoss()



## TASK 2 — BUILD: regularisation variants and EarlyStopping


Small CNN variant with configurable regularisation.


Monitor validation loss and stop when it plateaus.


In [ ]:
def build_variant(use_dropout: bool, use_bn: bool) -> nn.Module:
    layers: list[nn.Module] = [nn.Conv2d(1, 32, 3, padding=1)]
    if use_bn:
        layers.append(nn.BatchNorm2d(32))
    layers += [nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1)]
    if use_bn:
        layers.append(nn.BatchNorm2d(64))
    layers += [
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(4),
        nn.Flatten(),
        nn.Linear(64 * 4 * 4, 64),
        nn.ReLU(),
    ]
    if use_dropout:
        layers.append(nn.Dropout(0.3))
    layers.append(nn.Linear(64, N_CLASSES))
    return nn.Sequential(*layers).to(device)


class EarlyStopping:

    def __init__(self, patience: int = 4, min_delta: float = 1e-3) -> None:
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float("inf")
        self.should_stop = False

    def step(self, val_loss: float) -> bool:
        # TODO: If val_loss is more than min_delta better than best_loss,
        # update best_loss and reset counter. Otherwise, increment counter
        # and set should_stop=True when counter >= patience.
        if val_loss < self.best_loss - self.min_delta:
            ____
        else:
            ____
            if ____:
                self.should_stop = True
        return self.should_stop



## TASK 3 — TRAIN: variant grid + full pipeline


In [ ]:
print("\n--- Regularisation grid (6 epochs each) ---")
variant_labels = [
    (False, False, "Baseline"),
    (True, False, "Dropout"),
    (False, True, "BatchNorm"),
    (True, True, "Dropout+BN"),
]

variant_val_histories: dict[str, list[float]] = {}
for use_do, use_bn, label in variant_labels:
    net = build_variant(use_do, use_bn)
    opt_ = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    val_curve: list[float] = []
    for _ in range(6):
        train_cnn_one_epoch(net, train_loader, opt_, criterion)
        val_curve.append(eval_cnn(net, test_loader, criterion))
    variant_val_histories[label] = val_curve
    print(f"  {label:<12}: val {val_curve[0]:.4f} -> {val_curve[-1]:.4f}")

assert (
    variant_val_histories["Dropout+BN"][-1]
    <= variant_val_histories["Baseline"][-1] + 0.2
), "Task 3: Dropout+BN should not be dramatically worse than baseline"

# Full pipeline: TriageCNN + AdamW + cosine + clip + early stop
print("\n--- Full pipeline: TriageCNN + AdamW + cosine + clip + early stop ---")
model = TriageCNN(n_classes=N_CLASSES, dropout_rate=0.3).to(device)
optimiser = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

n_epochs = 12
# TODO: Create a cosine annealing scheduler with T_max=n_epochs, eta_min=1e-6.
scheduler = ____
# TODO: Create an EarlyStopping instance with patience=4, min_delta=1e-3.
stopper = ____

history = {"train_loss": [], "val_loss": [], "lr": [], "grad_norm": []}
for epoch in range(n_epochs):
    # TODO: Call train_cnn_one_epoch with clip_value=1.0 so the shared helper
    # applies gradient-norm clipping at 1.0.
    train_loss, grad = ____
    val_loss = eval_cnn(model, test_loader, criterion)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(scheduler.get_last_lr()[0])
    history["grad_norm"].append(grad)
    scheduler.step()
    print(
        f"  Epoch {epoch + 1:>2}/{n_epochs}: "
        f"train={train_loss:.4f} val={val_loss:.4f} "
        f"lr={history['lr'][-1]:.6f} grad={grad:.3f}"
    )
    if stopper.step(val_loss):
        print(f"  -> early stop triggered at epoch {epoch + 1}")
        break

assert stopper.best_loss < math.inf, "Task 3: early stopping must record a best loss"
assert all(
    g > 0 for g in history["grad_norm"]
), "Task 3: gradient norms must be positive"
print("\n[ok] Checkpoint passed — full training pipeline complete\n")



## TASK 4 — VISUALISE


In [ ]:
fig_variants = viz.training_history(variant_val_histories, x_label="Epoch")
fig_variants.update_layout(title="Regularisation Variants — Validation BCE")
fig_variants.write_html(OUTPUT_DIR / "05_variants_val.html")

fig_full = viz.training_history(
    {
        "Train BCE": history["train_loss"],
        "Val BCE": history["val_loss"],
        "Grad Norm": history["grad_norm"],
    },
    x_label="Epoch",
)
fig_full.update_layout(title="Full Pipeline — Loss and Gradient Norm")
fig_full.write_html(OUTPUT_DIR / "05_full_pipeline.html")
print("[viz] 05_variants_val.html + 05_full_pipeline.html saved")



## TASK 5 — APPLY: DSO National Laboratories Singapore — ONNX Export


In [ ]:
# SCENARIO: DSO runs imagery triage on edge devices with 512MB RAM and
# no Python runtime. PyTorch's 1.8GB footprint doesn't fit; ONNX Runtime
# (48MB, C++) does. Kailash-ml's OnnxBridge wraps the conversion,
# validates the exported model, and records the export for governance.

print("--- ONNX export via kailash-ml OnnxBridge + torch.onnx ---")

# Step 1: Compatibility check with the kailash-ml engine layer.
# TODO: Instantiate an OnnxBridge().
bridge = ____

compat = bridge.check_compatibility(model, framework="pytorch")
print(f"Compatible:  {compat.compatible}")
print(f"Confidence:  {compat.confidence}")

# Step 2: Export via torch.onnx.export (OnnxBridge.export natively handles
# sklearn/lightgbm; PyTorch models go through torch.onnx). Kailash-ml's
# InferenceServer (Module 5) consumes the resulting .onnx file.
onnx_output = OUTPUT_DIR / "triage_cnn.onnx"
model.eval()
dummy_input = torch.from_numpy(X_test_np[:1]).to(device)

# TODO: Call torch.onnx.export(model, dummy_input, str(onnx_output),
#   input_names=["image"], output_names=["logits"],
#   dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
#   opset_version=17)
____

export_size_bytes = onnx_output.stat().st_size
print(f"Exported:    {onnx_output.name}")
print(f"Model size:  {export_size_bytes / 1024:.1f} KB")

# Step 3: Validate against the PyTorch original using ONNX Runtime.
import numpy as _np
import onnxruntime as _ort

session = _ort.InferenceSession(str(onnx_output))
sample = X_test_np[:10]
with torch.no_grad():
    torch_out = model(torch.from_numpy(sample).to(device)).cpu().numpy()

# TODO: Run the ONNX session on the numpy sample and measure the max
# absolute difference from torch_out.
# Hint: session.run(None, {"image": sample})[0]  -> numpy array
onnx_out = ____
max_diff = float(_np.max(_np.abs(torch_out - onnx_out)))
print(f"Max diff (torch vs onnx): {max_diff:.8f}")

assert compat.compatible, "Task 5: TriageCNN must be ONNX-compatible"
assert onnx_output.exists(), "Task 5: ONNX file must be written"
assert (
    max_diff < 1e-4
), f"Task 5: ONNX output must match PyTorch within 1e-4, got {max_diff}"
print("\n[ok] Checkpoint passed — model exported to ONNX and validated\n")

# BUSINESS IMPACT:
#   PyTorch runtime 1.8GB -> ONNX Runtime 48MB. 7x reduction in inference
#   cost per edge node, same numerical output within 1e-4 tolerance.



## REFLECTION


[x] Compared four regularisation variants on identical budgets
  [x] Built EarlyStopping with patience + min_delta
  [x] Wired cosine decay + gradient clipping + early stopping together
  [x] Exported the trained model to ONNX via kailash-ml OnnxBridge

  This completes Exercise 8 and Module 4. Module 5 takes these blocks
  to real Fashion-MNIST, CIFAR-10, and transfer-learning backbones.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

